# Project Final Report

### Due: Midnight on April 27th (2-hour grace period) — 50 points  

### No late submissions will be accepted.


## Overview

Your final submission consists of **two components**:


### 1. Team Final Report Notebook [50 pts]

Complete all sections of this notebook to document your final decisions, results, and broader context.

You will write a **technical report** following standard conventions. Useful references include:
- [CMU guide to structure](https://www.stat.cmu.edu/~brian/701/notes/paper-structure.pdf)
- [Data science report example](https://www.projectpro.io/article/data-science-project-report/620)
- The Checklist in this week’s Blackboard Lesson (aligned with HOML)

Your audience is **technically literate but unfamiliar with your work**—for example, your manager or other data scientists. Your report should be clear, precise, and well-organized, combining explanation, visualizations, and interpretation.

This Final Report is distinct from Milestone 2:

- **Milestone 2** serves as a repository of your working code and experiments  
- **This Final Report** presents a clear, structured summary of your project for a professional audience  

> **Important:**
> - Do **not** assume that readers of this report are familiar with Milestone 2. Your report should stand on its own.
> - Do not include full code or code cells in this notebook. All code was submitted in Milestone 2. This report should focus on explanation, results, and interpretation.
> - **Do not add, delete, or move cells in this notebook.** Each answer must be written entirely within its assigned Markdown cell.
> - All answers should be inserted directly under the appropriate `Answer:` prompt. Delete the sentence "Replace this sentence with your answer" and replace it with your response.
> - You may use any appropriate Markdown formatting (paragraphs, lists, tables, inserted graphics, LaTeX, etc.).
> - Submit this notebook as a group via your team leader’s Gradescope account.


### 2. Individual Assessment [0 points]

Each team member must **individually** complete the Final Project Individual Assessment Form (similar to Milestone 1), sign it, and upload it via their own Gradescope account.

**Due:** May 5th @ 2:00 AM



## Submission Checklist

- Final Report Notebook — submitted by team leader  
- Individual Assessment Form — submitted by each team member  



## 1. Executive Summary [4 pts]

Write a 300–400 word executive summary for a **non-technical audience**, such as business stakeholders in a real estate company.

Your summary should explain:
- the objective of the project
- key insights about the data
- the most important findings, including a plain-language description of model performance
- a clear recommendation or takeaway

Avoid technical detail and jargon. Focus on what matters and why. 

**1.1 Answer:**  
Replace this sentence with your answer. 

## 2. Introduction [3 pts]

Introduce the topic, context, and goals of your project.

You may imagine that this project was completed for a **real estate company with a small in-house data science team**.

Include all of the following:

- Clearly introduce the topic and context of your project
- Describe the problem you are addressing (the problem statement) and the overall motivation for solving it
- Clearly state the objectives and goals of your analysis (as different from the motivation)

**2.1 Answer:**  

We have developed a model to predict property tax value assessments based on
easily available data on each property’s physical features. If we do not offer property tax
value estimates to clients, we are less effective in our sales and client satisfaction, as
clients cannot accurately evaluate whether a property is within their budget. Using this
model, we can include the tax value estimate as part of the monthly payment estimate
we report to clients, and they will be able to make a more informed decision on whether
a property is right for them. As an additional benefit, the data science team can use this
model internally to flag discrepancies between our own tax value estimates and any we
find online, which may signal dirty data.


## 3. Data Description [3 pts]

Describe the dataset used in your analysis.

Include:
- the source of the dataset
- the number of observations (samples) and features
- the types of variables
- the target variable
- missing values or other important issues in the raw data

**3.1 Answer:**  

This dataset is sourced from a dataset Zillow publicly released in 2017. The initial
subset of the data had 54 feature columns, one target column, and over 77k samples.


The features (variables) include physical features about the property such as: the
location of the property in latitude and longitude, location per zip code and county ID,
the house structure with number of bedrooms and bathrooms, and features of the house
such as pools, fireplaces, and garages. These units for these columns include
continuous quantitative measurements, counts, categories represented by numbers,
and some IDs in the form of discrete numerical numbers. The target variable is originally
the column called “taxvaluedollarcnt” in the dataset which is the continuous,
quantitative annual tax value in U.S. dollars and cents.
The raw data had a couple issues. There were many nulls, as 20 columns had over
90% null values, and 35 samples (0.05%) had a null target value. Some data did not
make sense across samples, such as a garage count of 1 but a garage area of 0. Also,
our team had to address some redundant and/or highly correlated features, often where
clusters of columns were closely related but not the exact same (for example multiple
columns about bathroom count). We also had to be cautious about the spread of data
within some columns that were skewed or required transformations so that we would
not falsely exclude valid data as outliers.

## 4. Methodology (What you did, and why) [20 pts]

Focus on the **process and your reasoning**, not the results.

Note: Each subsection (e.g., 4.1, 4.2, etc.) must be answered in its own Markdown cell.  Each subsection is worth 5 points. 

### 4.1 Analytical Framework

Describe your overall approach.

Include:
- your overall framework
- use of validation curves
- choice of MAE or RMSE (as appropriate) as the primary error metric

**4.1 Answer:**  
Our analytical framework followed a staged pipeline: 

(1) baseline modeling with default hyperparameters,   
(2) feature engineering,   
(3) feature selection,   
(4) hyperparameter tuning, and   
(5) final held-out test evaluation. We held out 20% of the ~77,000 cleaned observations as a test set, which was used exactly once — at the very end — to prevent any leakage from test-set performance into modeling decisions. 


For cross-validation, we used RepeatedKFold with 5 folds and 5 repeats (25 total folds) throughout the pipeline. This choice provided stable mean and standard deviation estimates for each model's generalization performance without depending on a single fold partition.


We selected MAE as our primary error metric rather than RMSE for two reasons. First, log-transforming the target variable (discussed in Section 4.2) converts MAE into a relative error metric: an MAE of ~0.47 in log-space means predictions are typically within a factor of ~1.6x of the true value, regardless of property price. RMSE, by contrast, would further amplify the already-reduced influence of outliers in log-space without adding interpretive benefit. Second, for a real estate business context, MAE is more interpretable: it represents the average prediction error rather than a root of squared deviations.


We used validation curves to diagnose overfitting and underfitting during hyperparameter tuning. For the Decision Tree, sweeping max_depth from 3 to unlimited confirmed that the default (unlimited) depth causes severe overfitting — training MAE near zero, high validation MAE. For Gradient Boosting, the interaction between n_estimators and learning_rate was visualized to confirm that more trees at lower learning rates improve generalization. These curves guided our hyperparameter grids before running GridSearchCV and RandomizedSearchCV.

### 4.2 Data Cleaning and Preprocessing

Describe how you prepared the data.

Include:
- issues in the raw data
- handling of missing values, outliers, inconsistencies
- key decisions and why
- what worked and what did not work

**4.2 Answer:**  

The Zillow housing data set provides many interesting choices for cleaning and preprocessing. We experimented with different ideas throughout the project. This data appears to be entered manually by those posting to the Zillow website, combined with data from the city and county. Because of this, there are frequent inconsistencies in the data.


To begin, we removed any houses without the target value or without square footage. We also removed features that contained many nulls, such as ‘buildingclasstypeid’, ‘architecturalstyletypeid’, ‘storytypeid’, and others. In this first look, we also removed ID features like ‘parcelid’, ‘fips’, and ‘censustractandblock’. A few of these features contain repeated data. For example, ‘fips’ is an alternative to county.


In our feature engineering, we decided to encode zip code with the target values, which is discussed in more detail in the next section. Thus, instead of dropping the houses with missing zip codes, we decided to impute them. We chose to impute using KNeighbors with latitude and longitude as our inputs. All houses at this point had latitude and longitude. This is likely because all Zillow pages include the physical location of the property for the map view. We imputed the missing city as well using the same technique. The ‘buildingqualitytypeid’ feature was likely to play a large role in predicting the tax values. Instead of imputing with the median or mean value, we again used KNeighbors. We saw that house condition was associated with zip code and decided that location may be the best predictor of house quality.


Many features required careful inspection of the underlying data. It appeared that missing garage data implied no garage. Missing story ID data appeared to imply only one story.


Lastly, some features proved more interesting to process. There were many different features related to both bathrooms and square footage. For many houses, not every variable related to square footage was present. We condensed both of these groups of features by taking the maximum of all related features.


In general, our cleaning and processing decisions appeared to have worked well. During feature selection, many of these features were high enough quality to be selected.


### 4.3 Feature Engineering

Describe your feature engineering.

Include:
- transformations or new features
- why you created them
- which were useful or not useful
- what worked and what did not work

**4.3 Answer:**  

During our preprocessing stage, we noted an extensive number of outliers for variables like the target and square footage. Very few of these properties appeared to be inconsistent data or errors; instead, there were just a significant number of properties that were orders of magnitude more expensive than a typical house. Instead of choosing to remove or cap the data, we took the log of both the tax assessment and square footage. We were then left with nearly symmetrical distributions that did not contain any significant outliers. This allowed us to include all of the original data while not having outliers overly influence the models. In addition, since the models are now minimizing log-scale error, they are closer to minimizing relative error. This is more natural: an error of $300,000 matters much more for a $500,000 property than one valued at $5,000,000. We feel this was the engineering decision that provided the largest boost to our models.


Another feature we engineered was zip code. There were too many zip codes for one-hot encoding to be effective. But upon closer inspection, mean and median tax values were highly dependent on zip code. This makes sense. In the Southern California area, where these houses are located, city and zip code are likely stronger determinants of house value than square footage. A small house in Beverly Hills or on the coast will be much more expensive than one in the many inland suburbs. To capture the importance of zip code, we chose to encode zip codes with the median tax value ranked 1 to 10. This engineering also proved to be impactful. It was frequently included during model selection. It may be possible that an advanced model could interpret latitude and longitude alone, but a simpler model could benefit from these zip rankings. We were careful to avoid any target leakage. The encoding was only done on the training data and this zip code mapping was applied to the test data. 


### 4.4 Model Selection

Describe how you selected your model.

Include:
- models you tried and why
- how you evaluated generalization
- hyperparameter tuning
- how you chose the final model

**4.4 Answer:**  

To select our final model, we compared lasso regression, a quick model that is
very interpretable; decision tree regressor, which is a great contrast to the lasso
model as it has a different way of learning the data and can capture non-linear
interactions; and gradient boosting trees, which is similar to decision tree
regressor but more powerful and accurate for mixed feature types, which is what our
data is.  

For each of these, we tuned the model’s hyperparameters by searching a range
of parameters, running the model along the range of possible hyperparameter
inputs, and then choosing the point of lowest error (measured in mean absolute
error). For lasso regression, we used a grid search with k-fold cross validation and
found the alpha with the lower error.  

For decision tree regressor, we used a grid search with k-fold validation to create
smaller ranges per parameter that are promising, then reran it again to find the best
combination of hyperparameters max depth, max leaf nodes, and minimum samples
per split.  

For gradient boosting trees, we used the same process to create smaller ranges
per parameter, but the final hyperparameter tuning on number of estimators, max
depth, minimum samples per split, minimum samples per leaf was completed with
randomized search to capture the interactions between these hyperparameters.
We concluded that gradient boosting trees is the best model based on the
following reasons. Throughout our entire experiment with running a baseline model
on the data, feature engineering for the model, selecting features for the model, and
tuning hyperparameters, this model consistently outperformed the lasso regressor
and decision tree regressor, with mean cross validation error approximately 27%
lower than the next best model.  We considered the trade-off between interpretability
and performance: Lasso offers coefficient-based interpretability, but the substantial
accuracy gap made Gradient Boosting the clear choice for a prediction task where
accuracy directly impacts business value. Its low standard deviation across folds
also gives confidence in the model’s stability and ability to generalize.

## 5. Results and Evaluation (What you found, and how well it worked) [16 pts]

Focus on **results, evidence, and interpretation**.

Note: Each subsection (e.g., 5.1, 5.2, etc.) must be answered in its own Markdown cell. 

### 5.1 Model Performance (6 points)

Briefly interpret what your metrics mean in practical terms.

Include:
- key metrics (e.g., RMSE, $R^2$)
- comparison across models
- comparison of training vs validation/test performance

**5.1 Answer:**  
Replace this sentence with your answer.

### 5.2 Visualizations (5 points)

Graphics should be made as screenshots and dragged into the Markdown cell. Do NOT add code cells to create graphics. Each visualization must be clearly labeled and explained in the text.

Include:
- relevant plots with titles and labels
- explanation of what each plot shows
- why each visualization matters

**5.2 Answer:**  

Here is a look at the original dataset. The following 2 images are 2 groups of
histograms, one with categorical features and one with numerical features, showing the
distribution of values for each column in the entire dataset. This is a good reference for
understanding the types of data and samples that are part of this model we developed.

![Histogram](histogram.png)

More specifically, here is a closer look at the distribution of the target variable. The
histogram on the left and boxplot on the right shows the distribution of the tax value of
the properties in our dataset. Notably, there is a skew of high tax value properties,
shown by the right tail on the histogram and the circles on the upper end of the boxplot.
Though properties with such a high tax value is less common in the dataset, it is valid
data and a situation we might encounter with clients with larger budgets. The shape of
this spread is the reason the target is log-transformed for the model.

![Plots](plots.png)


The following is a plot of each model and the error when run on training data as a
baseline. In green we see the gradient boosting regressor model has the best accuracy
(lowest mean absolute error) and most stability (lower standard deviation of error)
compared to the other 2 models. This indicates the gradient boosting regressor is the
best fit for our data, even at a baseline.

![Models Comparision](model_comparison.png)


While the gradient boosting regressor model automatically chooses to keep the most
important features, this visual gives us an insight into what the model considers to be

the most important factors when estimating tax value. It aligns with industry knowledge
and seems accurate.
For practical use, in cases when monthly budget is the largest concern for clients, we
can see whether they are flexible in these features. The most important features are:
house size, zip code, total calculated bathrooms per house, year built, and location by
latitude. These are the features that are likely to have the largest impact on property tax
value and monthly cost of property.

![Feature Importance](feature_importance.png)

After data was engineered and hyperparameters tuned to optimize model accuracy, the
final models’ performance can be found below. Once again,

https://github.com/waysnyder/Module-3-
Assignments/blob/main/Final_Project_Report.ipynb
https://github.com/kumargaurav-bu-
edu/bu_shared_projects/blob/semester2_dx603/milestone_2/Milestone_02.ipynb


### 5.3 Error Analysis (5 points)

Include:
- patterns in residuals or prediction errors
- overprediction or underprediction
- outliers or unusual observations
- anything surprising or worth improving

**5.3 Answer:**  
Our final Gradient Boosting model achieved a test MAE of 0.4715 in log-space, corresponding to $191,635 on average per property — a 57% improvement over the $441,420 average error from our initial baseline. Despite this improvement, several error patterns are worth examining.

**How prediction errors behave across price ranges** 
- Because we trained the model to predict log-transformed property values, prediction errors are more consistent across different price levels — the model does not make proportionally larger mistakes simply because a property is expensive. However, when we convert back to actual dollar amounts, the errors are still larger in absolute terms for high-value properties. For example, a 1.6x average error means a $300K mistake on a $500K home, but a $1.2M mistake on a $2M luxury property. In other words, the model works best for typical mid-range properties and is less reliable at the extremes.

**Where the model over- or under-predicts** 
- Location was the second most important signal in the model (30% of predictive power), just behind property size (41%). The model assigns higher predicted values to properties in high-value zip codes, which is generally correct. However, this creates predictable blind spots: the model is likely to under-predict value for an unusually well-maintained or renovated home in a modest neighborhood, and to over-predict for a distressed or unusual property in a typically expensive area.

**Hard cases and outliers** 
- Even after cleaning, the dataset spans a very wide price range. The model struggles most with luxury or unusual properties: these appear rarely in training data, and their value often depends on features we did not capture — such as ocean views, architectural style, or recent renovation history. Properties with uncommon use types (e.g., non-standard residential classifications) are also harder to price accurately.

**Surprising findings** 
- One unexpected result was that adding or removing features had almost no effect on Gradient Boosting's accuracy — the model performed equally well with 15 carefully selected features or the full feature set. This tells us the model is forgiving of extra information, but it also means we cannot easily trace errors back to any one feature. Separately, the Decision Tree model — even after tuning — remained noticeably less consistent than Gradient Boosting: its error varied more from one cross-validation fold to the next, confirming that single-tree models are more sensitive to which specific data they are trained on.

**What we would do next** 
- Plotting prediction errors against property price, location, and size would help pinpoint exactly where the model falls short. Breaking down error by price tier (e.g., under $300K, $300K–$700K, over $700K) or by county would reveal which market segments need further attention and whether a targeted sub-model might improve accuracy for high-value properties.

## 6. Conclusion [4 pts]

Summarize your findings and implications.

Include all the following:

- Clearly state your main findings and how they address your original objectives
- Highlight any business or practical implications of your findings 
- Discuss the limitations and constraints of your analysis clearly and transparently
- Suggest potential improvements or future directions
- Conclude with a final recommendation addressing the business objective

**6.1 Answer:**  
Replace this sentence with your answer.
